# Track 2 — Neural Alignment Evaluation

This notebook evaluates ONNX vision models for neural alignment with mouse visual cortex. For each model it measures how well the model's internal representations predict single-neuron activity recorded during the MouseAI competition.

**What this notebook does:**
- Runs each ONNX model on the competition stimulus frames
- Fits a linear readout (Ridge regression) from model activations to spike trains
- Reports **track2_score**: mean Pearson correlation between predicted and observed neural responses

**Data required:**
- Preprocessed session bundles — set `DATA_DIR` in the User Settings cell
- ONNX model file(s) — place models under `MODEL_DIR` and set `MODEL_NAME`

**Expected runtime (CPU):**
- Single model, 3 sessions: ~30–90 min · All 4 baselines: ~2–6 hours
- `MAX_LAYERS_PER_PASS = 8` is suitable for 16 GB RAM

**Output:** `track2_results.csv` in `OUTPUT_DIR` — one row per model with `track2_score` and metadata

| Setting | Options |
|---|---|
| `SESSION_SET` | `"official_3session"` — 3 original competition sessions · `"full_9session"` — all 9 released sessions |
| `MODEL_MODE` | `"single_model"` — evaluate `MODEL_NAME` only · `"all_models"` — every `.onnx` found under `MODEL_DIR` |
| `DEBUG` | `False` for a real run · `True` for a quick smoke test (scores not meaningful) |

In [ ]:
# ── User Settings ─────────────────────────────────────────────────────────────
from pathlib import Path

# Which recording sessions to use:
#   "official_3session"  →  3 sessions from the original competition evaluation
#   "full_9session"      →  all 9 released sessions (3 original + 6 additional)
SESSION_SET = "official_3session"

# Which model(s) to evaluate:
#   "single_model"  →  evaluate only MODEL_NAME (fastest; good starting point)
#   "all_models"    →  evaluate every .onnx file found under MODEL_DIR
MODEL_MODE = "single_model"
MODEL_NAME = "Baseline_NatureCNN"   # used when MODEL_MODE == "single_model"
                                     # must match the parent folder name under MODEL_DIR

# Evaluation options:
DEBUG           = False  # True = quick smoke test (scores are not meaningful)
FORCE_RECOMPUTE = False  # True = re-score even if a result already exists

# Memory / speed — higher is faster but uses more RAM; does NOT affect scores:
#   8 GB → 2    16 GB → 8    32 GB → 32    64 GB → 64
MAX_LAYERS_PER_PASS = 8

# Paths — defaults work out-of-the-box within the released package.
# Override only if your data or models are stored elsewhere.
ROOT = Path().resolve()

DATA_DIR = Path("./data")
MODEL_DIR  = ROOT / "models"
OUTPUT_DIR = ROOT / "results" / "track2_eval"

In [ ]:
# ── Imports and internal configuration ────────────────────────────────────────
import os, sys, time
import numpy as np
import pandas as pd

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.utils              import save_pkl, load_pkl
from src.data_loading       import load_all_bundles
from src.regression_readout import run_srp_ridge_cv


def find_onnx_models(search_dir: Path) -> pd.DataFrame:
    """
    Recursively find all .onnx files under search_dir.

    Returns a DataFrame with columns:
      model_name  — parent folder name if the file is inside a subfolder,
                    otherwise the file stem
      model_path  — absolute path to the .onnx file
    """
    search_dir = Path(search_dir)
    records = []
    for onnx_path in sorted(search_dir.rglob("*.onnx")):
        name = (
            onnx_path.parent.name
            if onnx_path.parent != search_dir
            else onnx_path.stem
        )
        records.append({"model_name": name, "model_path": str(onnx_path)})
    return pd.DataFrame(records)


# ── Session stems ──────────────────────────────────────────────
_SESSION_STEMS = {
    "official_3session": [
        "tigre569_p2s38_mousevAI_perturbs_preprocessed",
        "tigre613_p2s23_mousevAI_perturbs_preprocessed",
        "tigre847_p2s8_mousevAI_perturbs_preprocessed",
    ],
    "full_9session": [
        "tigre569_p2s38_mousevAI_perturbs_preprocessed",
        "tigre613_p2s23_mousevAI_perturbs_preprocessed",
        "tigre847_p2s8_mousevAI_perturbs_preprocessed",
        "tigre569_p2s28_mousevAI_perturbs_preprocessed",
        "tigre569_p2s35_mousevAI_perturbs_preprocessed",
        "tigre613_p2s21_mousevAI_perturbs_preprocessed",
        "tigre613_p2s22_mousevAI_perturbs_preprocessed",
        "tigre847_p2s5_mousevAI_perturbs_preprocessed",
        "tigre847_p2s9_mousevAI_perturbs_preprocessed",
    ],
}

# ── Evaluation hyperparameters (fixed — do not change) ────────
_SCORING_CFG = {
    "alpha":               1.0,    # Ridge regularisation strength
    "n_folds":             5,      # cross-validation folds (contiguous time blocks)
    "epsilon":             0.2,    # Johnson–Lindenstrauss random projection bound
    "batch_size":          128,    # ONNX inference batch size
    "max_layers_per_pass": MAX_LAYERS_PER_PASS,
    "visual_encoder_only": True,   # restrict to visual-encoder layers only
    "debug_frames":        200 if DEBUG else 0,
    "debug_neurons":       100 if DEBUG else 0,
}

In [ ]:
# ── Resolve paths and validate ────────────────────────────────────────────────
if SESSION_SET not in _SESSION_STEMS:
    raise ValueError(
        f"SESSION_SET must be 'official_3session' or 'full_9session', got {SESSION_SET!r}"
    )

stems         = _SESSION_STEMS[SESSION_SET]
session_paths = [str(DATA_DIR / f"{s}.npz") for s in stems]

RESULTS_DIR = OUTPUT_DIR / f"{MODEL_MODE}_{SESSION_SET}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

missing = [p for p in session_paths if not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        f"{len(missing)} session file(s) not found under:\n  {DATA_DIR}\n\n"
        "Missing:\n" + "\n".join(f"  {p}" for p in missing)
        + "\n\nCheck that DATA_DIR points to the released preprocessed dataset directory."
    )
if not MODEL_DIR.exists():
    raise FileNotFoundError(
        f"MODEL_DIR not found: {MODEL_DIR}\n"
        "Create the directory and place your .onnx model file(s) inside it."
    )

print(f"{'='*58}")
print(f"  Track 2 Neural Alignment Evaluation")
print(f"{'='*58}")
print(f"  Sessions   : {SESSION_SET}  ({len(stems)} sessions)")
print(f"  Models     : {MODEL_MODE}" + (f" — {MODEL_NAME}" if MODEL_MODE == "single_model" else ""))
print(f"  Data dir   : {DATA_DIR}")
print(f"  Models dir : {MODEL_DIR}")
print(f"  Output dir : {RESULTS_DIR}")
if DEBUG:
    print(f"\n  NOTE: DEBUG=True — scores will not be meaningful.")
print(f"{'='*58}")

---
## Evaluation method

**Linear readout from ONNX model activations**

For each model and recording session:
1. The model processes the competition stimulus frames (invalid frames excluded), using the visual-encoder layers only (before any action or policy head).
2. A Sparse Random Projection (ε = 0.2) reduces each layer's activation to a compact representation.
3. A Ridge regression model (α = 1.0) fits a linear mapping from model activations to single-neuron spike trains.
4. Pearson correlation is computed between predicted and observed spike trains on held-out time windows (5-fold cross-validation with contiguous time blocks; 2-second boxcar smoothing applied before correlation).

**track2_score** is the mean Pearson correlation across all evaluation neurons, using each session's best-predicting layer.

In [ ]:
# ── Load recording sessions ────────────────────────────────────────────────────
print("Loading recording sessions...\n")
bundles = load_all_bundles(session_paths)

print(f"\n{'─'*60}")
print(f"  {'Session':<44} {'Frames':>7} {'Neurons':>8}")
print(f"{'─'*60}")
total_neurons = 0
for b, stem in zip(bundles, stems):
    print(f"  {stem:<44} {b['T']:>7,} {b['N']:>8,}")
    total_neurons += b["N"]
print(f"{'─'*60}")
print(f"  Total evaluation neurons: {total_neurons:,}")

In [ ]:
# ── Discover models ────────────────────────────────────────────────────────────
df_all = find_onnx_models(MODEL_DIR)
if df_all.empty:
    raise FileNotFoundError(
        f"No .onnx files found under {MODEL_DIR}.\n"
        "Place your model file(s) inside a subfolder (e.g. models/Baseline_NatureCNN/)."
    )

if MODEL_MODE == "single_model":
    df_models = df_all[df_all["model_name"] == MODEL_NAME].reset_index(drop=True)
    if df_models.empty:
        avail = df_all["model_name"].tolist()
        raise ValueError(
            f"MODEL_NAME='{MODEL_NAME}' not found under {MODEL_DIR}.\n"
            f"Available models: {avail}"
        )

elif MODEL_MODE == "all_models":
    df_models = df_all.reset_index(drop=True)

else:
    raise ValueError(
        f"MODEL_MODE must be 'single_model' or 'all_models', got {MODEL_MODE!r}"
    )

print(f"Models to evaluate: {len(df_models)}\n")
df_models

In [ ]:
# ── Score models ───────────────────────────────────────────────────────────────
# For each model: extract ONNX layer activations → Ridge regression → Pearson r
#
# Resume-safe: existing result files are skipped unless FORCE_RECOMPUTE=True.
# Re-run this cell after an interruption to pick up where it left off.

n_total  = len(df_models)
_skipped = []

for i, (_, row) in enumerate(df_models.iterrows(), 1):
    model_name = row["model_name"]
    model_path = row["model_path"]
    out_path   = RESULTS_DIR / f"{model_name}.pkl"

    print(f"\n[{i}/{n_total}] {model_name}")

    if out_path.exists() and not FORCE_RECOMPUTE:
        print("  skip — result already exists  (set FORCE_RECOMPUTE=True to re-score)")
        continue

    if out_path.exists() and FORCE_RECOMPUTE:
        out_path.unlink()
        print("  removed existing result (FORCE_RECOMPUTE=True)")

    _t0 = time.time()
    result = run_srp_ridge_cv(
        model_path=model_path,
        session_bundles=bundles,
        cfg=_SCORING_CFG,
        debug=DEBUG,
    )
    _elapsed = round(time.time() - _t0, 1)

    if result is None:
        print("  FAIL — model skipped (see warnings above)")
        _skipped.append(model_name)
        continue

    result["model"]           = model_name
    result["model_path"]      = model_path
    result["runtime_seconds"] = _elapsed
    save_pkl(result, out_path)
    print(f"  done in {_elapsed:.0f}s → {out_path.name}")

print(f"\n{'─'*50}")
if _skipped:
    print(f"Completed with {len(_skipped)} failure(s): {_skipped}")
else:
    print(f"All {n_total} model(s) scored successfully.")

In [ ]:
# ── Build results table ────────────────────────────────────────────────────────
# Reads all scored .pkl files and computes track2_score for each model.
#
# track2_score = mean Pearson correlation between model-predicted and observed
# spike trains, at each session's best-predicting layer, averaged across all
# evaluation neurons.

records = []
for pkl_path in sorted(RESULTS_DIR.glob("*.pkl")):
    try:
        d = load_pkl(pkl_path)
    except Exception as e:
        print(f"  WARNING: could not read {pkl_path.name}: {e}")
        continue

    model_name = d["model"]

    session_scores = []
    total_n = 0
    for mb in d["per_mouse"].values():
        scores = np.asarray(mb["scores"])              # [L, N]
        best_L = int(np.nanargmax(np.nanmean(scores, axis=1)))
        session_scores.extend(scores[best_L].tolist())
        total_n += scores.shape[1]

    if not session_scores:
        print(f"  WARNING: no valid scores for {model_name} — skipping")
        continue

    records.append({
        "model_name":      model_name,
        "model_path":      d.get("model_path", ""),
        "session_set":     SESSION_SET,
        "n_sessions":      len(stems),
        "n_neurons":       total_n,
        "track2_score":    round(float(np.nanmean(session_scores)), 6),
        "runtime_seconds": round(d.get("runtime_seconds", float("nan")), 1),
    })

if not records:
    raise RuntimeError(
        f"No results found in {RESULTS_DIR}.\n"
        "Run the scoring cell above first."
    )

df_results = (
    pd.DataFrame(records)
    .sort_values("track2_score", ascending=False)
    .reset_index(drop=True)
)

out_csv = RESULTS_DIR / "track2_results.csv"
df_results.to_csv(out_csv, index=False)
print(f"Results saved → {out_csv}\n")
df_results[["model_name", "track2_score", "n_sessions", "n_neurons", "runtime_seconds"]]

In [ ]:
# ── Final summary ──────────────────────────────────────────────────────────────
print(f"{'='*58}")
print(f"  Track 2 Evaluation — Results")
print(f"{'='*58}")
print(f"  Session set   : {SESSION_SET}")
print(f"  Sessions      : {len(stems)}")
print(f"  Total neurons : {total_neurons:,}")
print()
print(f"  {'Model':<40} {'track2_score':>12}")
print(f"  {'─'*40}   {'─'*12}")
for _, row in df_results.iterrows():
    rt = f"  ({row['runtime_seconds']:.0f}s)" if not np.isnan(row["runtime_seconds"]) else ""
    print(f"  {row['model_name']:<40} {row['track2_score']:>12.4f}{rt}")
print()
print(f"  Full results: {out_csv}")
print(f"{'='*58}")